In [1]:
import os
import sys

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path: sys.path.append(module_path)
from ytmusic_library import YTMusicPlaylists

import ytmusicapi as ytmusicapi

print(f'Using ytmusicapi version: {ytmusicapi.__version__}')

from ytmusic_library import YTMusicPlaylists

RUN_API_AUTH_TEST = True
HEADER_FILE = '../oauth.json'
PLAYLIST_TSV_DIR = '../playlists/'
# PLAYCOUNT_FILE='../playlists/_ytmusic_lastfm_match_id_map.tsv'
NOT_LIKE_FILE  = '../playlists/_not_liked_tracks.tsv'

Y = YTMusicPlaylists(header=HEADER_FILE, playlist_tsv_dir=PLAYLIST_TSV_DIR)
if RUN_API_AUTH_TEST: Y.test_ytmusic_api()
print(f"Loaded {len(Y.playlists['title'].unique())} playlists")

Using ytmusicapi version: 1.3.1
Using header file: ../oauth.json
Test Passed in 4.46 seconds
Using ytmusicapi version: 1.3.1
Loaded 463 playlists


## TODO replace clean_up_all_radio_playlists with a clean_up_all_playlists (no radio)

* make sure it replicates clean_up_all_radio_playlists plus handles tasks for no radio


# Process Each Playlist

##### add playlist to super playliusts if exist see pdf
##### auto gnerate some date based like playlsiysd
##### TODO move based on playcount (if not LIKE infer NOT_LIKE based on large playcount)
##### TODO make into script, run monthly



In [2]:
playlists_kinds = {k: set() for k in Y._valid_playlist_kinds}
for i, p in Y.playlists.iterrows():
  print(p.title)

AttributeError: 'YTMusicPlaylists' object has no attribute '_valid_playlist_kinds'

In [16]:
# Skip playlistes inferred as these kind
DRY_RUN = True  # make sure all NOT OK is fine

VPRINT = False  # Verbose printing
WARN_PRINT = True  # Print warnings

# Options for checking inferred playlist kind
SKIP_PLAYLIST_KINDS = ('SKIP', 'ALBUM', 'YT_GENERATED')
LIKE_MIN_LIKE_PCT = 80
NOTLIKE_MAX_LIKE_PCT = 20
RADIO_MAX_LIKE_PCT = 50

# Options for processing playlists
MIN_RADIO_LIKE_TO_SPLIT = 10
DUPLICATE_THRESHOLD = 3  # was 4 first run

playlists_kinds = {k: set() for k in Y._valid_playlist_kinds}
for i, p in Y.playlists.iterrows():
    if p.title.startswith('zz not like'):
        continue

    if VPRINT:
        print(100*'=' + f'\nPlaylist: {p.title} ({p.playlistId})',
              f'has {p.count} tracks')

    """Potentially skip playlist"""
    # Infer playlist kind from the title, default to LIKE if nothing inferred
    pl_kind = Y.infer_playlist_kind(p)
    if not pl_kind:
        pl_kind = 'LIKE'

    # Decide to skip playlist based on playlist kind
    playlists_kinds[pl_kind].add(p.title)
    if pl_kind in SKIP_PLAYLIST_KINDS:
        print(f'SKIPPING playlist: {p.title} as it is',
              f'a kind flagged for skipping: {pl_kind}')
        continue

    """Query playlist tracks then potentially skip"""
    # Query playlist tracks and other metadata
    p_info = Y.playlist_get_info(
        p.playlistId, playlist_limit=Y.playlist_limit).copy()
    if p_info['trackCount'] == 0:
        print('No tracks in playlist')
        continue
    # Check max length of playlist
    if len(p_info['tracks']) >= Y.playlist_limit:
        print(f'SKIPPING playlist: {p.title} which has',
              f'{Y.playlist_limit} or more tracks ({len(p_info["tracks"])})')
        continue

    # Check playlist privacy
    if p_info['privacy'] == 'PUBLIC':
        print(f'SKIPPING playlist: {p.title} which has',
              f'privacy: {p_info["privacy"]}')
        continue
    elif p_info['privacy'] == 'UNLISTED' and WARN_PRINT:
        print(f'WARNING {p_info["privacy"]} playlist: {p.title}')

    # Get ratings for playlist tracks
    ratings = {k: set() for k in Y._valid_ratings}
    for track in p_info["tracks"]:
        if track["likeStatus"] not in ratings.keys():
            ratings['NONE'].add(track["videoId"])
        else:
            ratings[track["likeStatus"]].add(track["videoId"])

    # See if playlist is correctly flagged as LIKE or RADIO
    like_percent = round(100*len(ratings["LIKE"])/len(p_info["tracks"]))
    if not Y._is_playlist_kind_ok(pl_kind, like_percent,
                                  LIKE_MIN_LIKE_PCT, NOTLIKE_MAX_LIKE_PCT,
                                  RADIO_MAX_LIKE_PCT):
        if WARN_PRINT:
            print(f'WARNING NOT OK {pl_kind} Playlist: {p.title}',
                  f'({like_percent}% liked) has: {len(ratings["LIKE"])} likes,',
                  f'{len(ratings["DISLIKE"])} dislikes,{len(ratings["INDIFFERENT"])}',
                  f'indifferent, {len(ratings["NONE"])} none')

    """Potentially alter playlist, or generate new playlists"""
    if DRY_RUN:
        continue

    # Remove duplicates from playlist
    new_pl_id = Y.playlist_remove_duplicates(
        p_info, duplicate_threshold=DUPLICATE_THRESHOLD, verbose=VPRINT)
    if p.playlistId != new_pl_id:
        p.playlistId = new_pl_id
        p_info = Y.playlist_get_info(
            new_pl_id, playlist_limit=Y.playlist_limit, use_cache=False)

    # Like all tracks in playlist if kind is LIKE
    if pl_kind == 'LIKE':
        Y.playlist_rate_all_songs(
            p_info, rating=pl_kind, skip_if_dislike=True, verbose=VPRINT)
        continue
    # Split radio playlist into LIKE vs RADIO
    elif pl_kind == 'INDIFFERENT':
        Y.clean_up_radio_playlist(
            p_info, verbose=VPRINT,  move_like=True, 
            min_num_like=MIN_RADIO_LIKE_TO_SPLIT,
            create_like_playlist=True,
            remove_dislike=True, remove_not_like=True
        )
        continue


Playlist folk 1960s: Rated 0 of 99 tracks as LIKE
Playlist Folk: Rated 0 of 200 tracks as LIKE
Playlist electronic witch_house: Rated 3 of 56 tracks as LIKE
Playlist electronic we are alone: Rated 0 of 49 tracks as LIKE
Playlist electronic uk: Rated 9 of 264 tracks as LIKE
Playlist electronic trance dj: Rated 2 of 27 tracks as LIKE
Not splitting playlist electronic soft pad radio,  not enough likes (0)
Playlist electronic soft pad: Rated 1 of 55 tracks as LIKE
Not splitting playlist electronic radio,  not enough likes (4)
Playlist electronic new indie beats: Rated 6 of 405 tracks as LIKE
Playlist electronic jaar: Rated 1 of 38 tracks as LIKE
Not splitting playlist electronic Innerwaves radio,  not enough likes (2)
Playlist electronic Innerwaves: Rated 0 of 32 tracks as LIKE
Not splitting playlist electronic indie radio,  not enough likes (1)
Playlist electronic indie essentials: Rated 0 of 52 tracks as LIKE
Not splitting playlist electronic House Special radio,  not enough likes (1)
Pl

# Todo use latest function to get all like playlists concat this with manual picks already in yt_lib, maybe combine  with radio map
```
  'Beats Without Rhymes like',
	'Beats indie Chill like',
	'Bossa Nova like',
	'Brass n chill',
	'Chillwave',
	'Electronic 2010s like',
	'Electronic Focus like',
	'Electronic House Special like',
	'Electronic Innerwaves like',
	'Folk like',
	'Future Bass Instrumentals like',
	'Grunge like',
	'Hip Hop 1990s like',
	'Hip Hop 2000s like',
	'Hip Hop Classic West Coast like',
	'Hip Hop Hits liked',
	'Hip hop It Was a Good Day like',
	'Indie 1990s Rock like',
	'Indie 2000s like',
	'Indie Mellow like',
	'Jukebox Vintage Party like',
	'Oldies like',
	'Post-Punk 1970s-1980s like',
	'Psychedelic Indie like',
	'Reggae Dub like',
	'Reggae like',
	'Rock & Roll Beatles like',
	'Rock 1967-1969 like',
	'Rock 1980s Pop  New Wave like',
	'Shoegaze Nu like',
	'Shoegaze',
	'Soul Classic Sunshine like',
	'Your Likes',
	'ambient Dream Pop Deep Sleep like',
	'ambient Indie synths',
	'ambient haunting harmonious like',
	'beats cosmic Slop like',
	'beats instrumental',
	'blues delta roots like',
	'blues',
	'doo wop like',
	'electronic Analog Grooves like',
	'electronic chill',
	'electronic dubstep uk like',
	'electronic indie essentials like',
	'electronic new indie beats',
	'electronic top',
	'electronic we are alone like',
	'folk 1960s like',
	'future bass',
	'future beats',
	'future funk airlines tracks like',
	'future garage',
	'futurebeat_rap',
	'futurebeats subreddit like',
	'garage rock',
	'goth 1980s like',
	'hiphop 80s like',
	'hiphop modern',
	'hiphop old school like',
	'hiphop wrist twistin like',
	'hiphop',
	'indie loose like',
	'indie pop Tar Beach Lullabies like',
	'indie',
	'jazz cool',
	'jazz gloom smooth like',
	'jazz noir',
	'jazz solo guitar like',
	'jazz',
	'jukebox 1950s like',
	'jukebox 1960s like',
	'jukebox elvis like',
	'my balls your chill',
	'nudisco',
	'pop singer songwriters like',
	'post rock slow core like',
	'psych rock modern',
	'psychedelic classic rock',
	'psychedelic rock subreddit like',
	'punk 1970s like',
	'reggae classic',
	'reggae modern',
	'rnb dj',
	'rock 1950s roots like',
	'rock 1960s classic',
	'rock 1970s classic like',
	'rock 1980s college radio like',
	'rock 1990s alternative like',
	'rock 2000s radio like',
	'rock Deep Cut Kick Back like',
	'rock Motorik like',
	'rock classic like',
	'rock instrumentals classic vintage like',
	'rock krautrock',
	'rock modern chill',
	'rock proto metal like',
	'rock surf like',
	'rock surf modern like',
	'soul 1960s like',
	'soul funk',
	'soul motown like',
	'trip hop like',
	'x_r.chillwave_tracks_radio like',
	'x_r.hiphop_tracks_radio like',
	'x_r.treemusic like',
	'y_1950s_thumbs_up',
	'y_1960s_thumbs_up',
	'y_1970s_thumbs_up',
	'y_1990s_thumbs_up',
	'y_2000s_thumbs_up',
	'y_2005s_thumbs_up',
	'y_2010_thumbs_up',
	'y_2012_thumbs_up',
	'y_2013_thumbs_up',
	'y_2014_thumbs_up',
	'y_2015_thumbs_up',
	'y_2016_thumbs_up',
	'y_2017_thumbs_up',
	'y_2018_thumbs_up',
	'y_2019_thumbs_up',
	'y_2020_thumbs_up',
	'z_dj_thumbs_up',
	'zz__thumbs_up'
```